# 📖 Notebook 3: Group Messaging

Welcome back! In the previous notebooks we explored **1:1 messaging** and **read receipts**. Now it’s time for the big one — **group messaging**. This is where things get interesting because sending a single message needs to reach *multiple* people.

## 🎯 What You’ll Learn

1. **Fan-out** — How one message becomes many deliveries
2. **Partitioning strategies** — Should we organize Redis pub/sub channels by user or by chat?
3. **Admin controls** — Adding and removing group members
4. **Scaling challenges** — What happens when groups get really big?

> 💡 **Real-world context:** WhatsApp groups can have up to 1,024 members. Our demo uses 100 as the limit, but the same principles apply at any scale.

## 🛠️ Setup

**1. Start the infrastructure** (if not already running):

```bash
cd system-designs/whatsapp
docker-compose up -d
```

**2. Select the correct kernel:**
- In VS Code, click the kernel picker (top-right of this notebook)
- Select the `.venv` kernel from this lab’s virtual environment
- If it doesn’t appear, reload VS Code (`Cmd+Shift+P` → “Reload Window”)

**3. Install dependencies** (if not already done):

```bash
uv sync
```

In [1]:
import psycopg2
import psycopg2.extras
import redis
import json
import time
import threading
from datetime import datetime
from tabulate import tabulate
from websockets.sync.client import connect as ws_connect

# ==== Connection Configuration ====
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "whatsapp_demo",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {"host": "localhost", "port": 6379, "decode_responses": True}
WS_URL = "ws://localhost:8765"

def get_db():
    """Get a fresh database connection."""
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    """Get a fresh Redis client."""
    return redis.Redis(**REDIS_CONFIG)

def query(sql, params=None):
    """Run a SELECT query and return rows as dictionaries."""
    conn = get_db()
    try:
        cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
        cur.execute(sql, params)
        rows = cur.fetchall()
        cur.close()
        return rows
    finally:
        conn.close()

def execute(sql, params=None):
    """Run an INSERT/UPDATE/DELETE query."""
    conn = get_db()
    try:
        cur = conn.cursor()
        cur.execute(sql, params)
        conn.commit()
        cur.close()
    finally:
        conn.close()

# ==== Test Connections ====
try:
    conn = get_db()
    conn.close()
    print("\u2705 PostgreSQL connected")
except Exception as e:
    print(f"\u274c PostgreSQL error: {e}")

try:
    r = get_redis()
    r.ping()
    print("\u2705 Redis connected")
except Exception as e:
    print(f"\u274c Redis error: {e}")

try:
    ws = ws_connect(WS_URL)
    ws.close()
    print("\u2705 WebSocket server reachable")
except Exception as e:
    print(f"\u274c WebSocket error: {e}")

✅ PostgreSQL connected
✅ Redis connected
✅ WebSocket server reachable


## 💡 Key Insight: 1:1 is Just a Special Case of Groups

Before we dive into groups, here’s a powerful design insight:

> **A 1:1 chat is simply a group with exactly 2 members.**

This means we don’t need separate code paths for 1:1 and group messaging! The same fan-out logic handles both.

```
┌──────────┬──────────────┬──────────┬───────────────────────┐
│ chat_id  │ name         │ is_group │ max_participants      │
├──────────┼──────────────┼──────────┼───────────────────────┤
│ 1        │ NULL         │ false    │ 2   ← 1:1 (Alice↔Bob) │
│ 2        │ NULL         │ false    │ 2   ← 1:1 (Alice↔Ch.)│
│ 3        │ Study Group  │ true     │ 100 ← Group chat      │
└──────────┴──────────────┴──────────┴───────────────────────┘
```

**Why this is smart:**
- ✅ **Same fan-out logic** — Send message → look up participants → deliver to each
- ✅ **Same inbox pattern** — Every recipient gets an inbox entry
- ✅ **Same sequence numbers** — Per-chat ordering works for both
- ✅ **Less code, fewer bugs** — One code path instead of two

The only differences for groups:
- Groups have a **name** (1:1 chats use `NULL`)
- Groups have **admin roles** (1:1 chats don’t need admins)
- Groups have a **higher member limit** (100 vs 2)

## 🔍 Section 1: Exploring Existing Groups

Our seed data includes a group called **“Study Group”** (chat_id = 3) with Alice as admin and Bob, Charlie, Diana as members. Let’s explore it.

In [2]:
# ==== Look at the Study Group ====
print("\U0001f4cb Chat Details:\n")
chats = query("SELECT id, name, is_group, max_participants, created_at FROM chats WHERE id = 3")
print(tabulate(chats, headers="keys", tablefmt="simple_grid"))

print("\n\n\U0001f465 Group Members:\n")
members = query("""
    SELECT cp.user_id, u.display_name, cp.role, cp.joined_at
    FROM chat_participants cp
    JOIN users u ON u.id = cp.user_id
    WHERE cp.chat_id = 3
    ORDER BY cp.role DESC, cp.joined_at
""")
print(tabulate(members, headers="keys", tablefmt="simple_grid"))

print("\n\n\U0001f4ac Existing Messages in Study Group:\n")
messages = query("""
    SELECT m.id, u.display_name AS sender, m.content,
           m.sequence_number, m.server_timestamp
    FROM messages m
    JOIN users u ON u.id = m.sender_id
    WHERE m.chat_id = 3
    ORDER BY m.sequence_number
""")
print(tabulate(messages, headers="keys", tablefmt="simple_grid"))

print(f"\n\U0001f4ca Study Group has {len(members)} members and {len(messages)} messages")

📋 Chat Details:

┌──────┬─────────────┬────────────┬────────────────────┬────────────────────────────┐
│   id │ name        │ is_group   │   max_participants │ created_at                 │
├──────┼─────────────┼────────────┼────────────────────┼────────────────────────────┤
│    3 │ Study Group │ True       │                100 │ 2026-04-20 04:13:56.498173 │
└──────┴─────────────┴────────────┴────────────────────┴────────────────────────────┘


👥 Group Members:

┌───────────┬────────────────┬────────┬────────────────────────────┐
│   user_id │ display_name   │ role   │ joined_at                  │
├───────────┼────────────────┼────────┼────────────────────────────┤
│         3 │ Charlie Brown  │ member │ 2026-04-20 04:13:56.498547 │
├───────────┼────────────────┼────────┼────────────────────────────┤
│         4 │ Diana Prince   │ member │ 2026-04-20 04:13:56.498547 │
├───────────┼────────────────┼────────┼────────────────────────────┤
│         2 │ Bob Smith      │ member │ 2026-04-20

## 🆕 Section 2: Creating a New Group

Let’s create a brand new group using the WebSocket protocol. **Alice** will create a group with **Bob** and **Charlie**.

**The create flow:**
```
Alice's Client          Server                    Database
     │                    │                          │
     ├── create_chat ────>│                          │
     │   participants:    │── INSERT into chats ────>│
     │   [1,2,3]         │── INSERT participants ──>│
     │   name: "Project" │── INSERT chat_sequences ->│
     │                    │                          │
     │<── chat_created ──┤                          │
     │    chat_id: N      │                          │
```

In [3]:
# ==== Create a new group as Alice (user_id=1) ====
print("\U0001f195 Creating a new group chat...\n")

ws = ws_connect(WS_URL)

# Step 1: Connect as Alice
ws.send(json.dumps({"type": "connect", "user_id": 1}))
response = json.loads(ws.recv())
print(f"1\ufe0f\u20e3  Connected as Alice: {response}")

# Step 2: Create a group with Alice(1), Bob(2), Charlie(3)
ws.send(json.dumps({
    "type": "create_chat",
    "participants": [1, 2, 3],
    "name": "Project Team"
}))
response = json.loads(ws.recv())
print(f"2\ufe0f\u20e3  Group created: {response}")

new_chat_id = response.get("chat_id")
ws.close()

time.sleep(0.3)  # let the server finish writing

# Step 3: Verify in the database
print(f"\n\U0001f4cb New Chat (id={new_chat_id}):\n")
chat_info = query(
    "SELECT id, name, is_group, max_participants FROM chats WHERE id = %s",
    (new_chat_id,)
)
print(tabulate(chat_info, headers="keys", tablefmt="simple_grid"))

print(f"\n\U0001f465 Participants:\n")
participants = query("""
    SELECT cp.user_id, u.display_name, cp.role
    FROM chat_participants cp
    JOIN users u ON u.id = cp.user_id
    WHERE cp.chat_id = %s
    ORDER BY cp.role DESC, cp.user_id
""", (new_chat_id,))
print(tabulate(participants, headers="keys", tablefmt="simple_grid"))

name = chat_info[0]['name']
print(f"\n\u2705 Group '{name}' created with {len(participants)} members!")

🆕 Creating a new group chat...

1️⃣  Connected as Alice: {'type': 'connected', 'user_id': 1}
2️⃣  Group created: {'type': 'chat_created', 'chat_id': 4}



📋 New Chat (id=4):

┌──────┬──────────────┬────────────┬────────────────────┐
│   id │ name         │ is_group   │   max_participants │
├──────┼──────────────┼────────────┼────────────────────┤
│    4 │ Project Team │ True       │                100 │
└──────┴──────────────┴────────────┴────────────────────┘

👥 Participants:

┌───────────┬────────────────┬────────┐
│   user_id │ display_name   │ role   │
├───────────┼────────────────┼────────┤
│         2 │ Bob Smith      │ member │
├───────────┼────────────────┼────────┤
│         3 │ Charlie Brown  │ member │
├───────────┼────────────────┼────────┤
│         1 │ Alice Johnson  │ admin  │
└───────────┴────────────────┴────────┘

✅ Group 'Project Team' created with 3 members!


## 📤 Section 3: Fan-Out — Sending to a Group

When Alice sends a message to “Study Group” (4 members), here’s what happens:

```
Alice sends: "Hey everyone! 👋"
         │
         ▼
   ┌───────────┐
   │  Server   │──── 1. Store message in messages table
   └───────────┘
         │
    Fan-Out!          2. Create inbox entries for EACH recipient
         │               (everyone except Alice)
         ├──────────────────┬──────────────────┐
         ▼                  ▼                  ▼
   ┌──────────┐      ┌──────────┐      ┌──────────┐
   │  Bob’s   │      │Charlie’s │      │ Diana’s  │
   │  inbox   │      │  inbox   │      │  inbox   │
   └──────────┘      └──────────┘      └──────────┘
         │                  │                  │
         ▼                  ▼                  ▼
   ┌──────────┐      ┌──────────┐      ┌──────────┐
   │  Redis   │      │  Redis   │      │  Redis   │
   │ user:2   │      │ user:3   │      │ user:4   │
   │ pub/sub  │      │ pub/sub  │      │ pub/sub  │
   └──────────┘      └──────────┘      └──────────┘
```

### ⚡ Write Amplification

This is called **fan-out on write** — it has a real cost:

| Group Size | Messages Sent | Inbox Rows Created | Redis Publishes |
|-----------|---------------|-------------------|----------------|
| 2 (1:1)  | 1             | 1                 | 1              |
| 5         | 1             | 4                 | 4              |
| 50        | 1             | 49                | 49             |
| 100       | 1             | 99                | 99             |

> 💡 **Write amplification** means one user action creates multiple database writes. A single message to a 100-person group creates **99 inbox rows**! This is the fundamental scaling challenge of group messaging.

In [4]:
# ==== Send a message to Study Group as Alice ====
print("\U0001f4e4 Sending a message to Study Group (chat_id=3)...\n")

# Count existing inbox entries for Study Group messages
before_count = query(
    "SELECT COUNT(*) as count FROM inbox WHERE message_id IN "
    "(SELECT id FROM messages WHERE chat_id = 3)"
)
print(f"\U0001f4ca Inbox entries before: {before_count[0]['count']}")

# Connect as Alice and send a message
ws = ws_connect(WS_URL)
ws.send(json.dumps({"type": "connect", "user_id": 1}))
connect_resp = json.loads(ws.recv())
print(f"\u2705 Connected as Alice")

ws.send(json.dumps({
    "type": "send_message",
    "chat_id": 3,
    "content": "Hey everyone! Let's study together tonight! \U0001f4da"
}))
ack = json.loads(ws.recv())
print(f"\u2705 Message sent! ACK: {ack}")
message_id = ack.get("message_id")

ws.close()
time.sleep(0.5)  # let the server finish the fan-out

# Check the inbox entries that were created
after_count = query(
    "SELECT COUNT(*) as count FROM inbox WHERE message_id IN "
    "(SELECT id FROM messages WHERE chat_id = 3)"
)
print(f"\U0001f4ca Inbox entries after:  {after_count[0]['count']}")
print(f"   \u2192 New entries created: {after_count[0]['count'] - before_count[0]['count']}")

print(f"\n\U0001f4ec Inbox entries for the new message (id={message_id}):\n")
inbox_rows = query("""
    SELECT i.id, u.display_name AS recipient, i.status, i.created_at
    FROM inbox i
    JOIN users u ON u.id = i.user_id
    WHERE i.message_id = %s
    ORDER BY u.display_name
""", (message_id,))
print(tabulate(inbox_rows, headers="keys", tablefmt="simple_grid"))

print(f"\n\U0001f4a1 Notice: Alice (the sender) does NOT have an inbox entry.")
print(f"   1 message \u2192 {len(inbox_rows)} inbox entries = fan-out to all members except sender!")

📤 Sending a message to Study Group (chat_id=3)...

📊 Inbox entries before: 3
✅ Connected as Alice
✅ Message sent! ACK: {'type': 'ack', 'message_id': 14, 'status': 'stored'}


📊 Inbox entries after:  6
   → New entries created: 3

📬 Inbox entries for the new message (id=14):

┌──────┬───────────────┬──────────┬────────────────────────────┐
│   id │ recipient     │ status   │ created_at                 │
├──────┼───────────────┼──────────┼────────────────────────────┤
│    9 │ Bob Smith     │ pending  │ 2026-04-20 04:20:21.798318 │
├──────┼───────────────┼──────────┼────────────────────────────┤
│   10 │ Charlie Brown │ pending  │ 2026-04-20 04:20:21.798318 │
├──────┼───────────────┼──────────┼────────────────────────────┤
│   11 │ Diana Prince  │ pending  │ 2026-04-20 04:20:21.798318 │
└──────┴───────────────┴──────────┴────────────────────────────┘

💡 Notice: Alice (the sender) does NOT have an inbox entry.
   1 message → 3 inbox entries = fan-out to all members except sender!


## 📨 Section 4: Receiving Group Messages

Now let’s see the other side — **Bob** and **Charlie** both receive Alice’s group message through Redis pub/sub. Each user has their own channel.

```
Redis pub/sub channels:
  user:2  ──→  Bob's WebSocket connection
  user:3  ──→  Charlie's WebSocket connection
  user:4  ──→  Diana's WebSocket (if she were online)
```

We’ll connect as Bob and Charlie, sync their pending messages, and ACK them.

In [5]:
# ==== Connect as Bob and Charlie, receive and ACK messages ====
print("\U0001f4e8 Connecting as Bob and Charlie to receive group messages...\n")

received_messages = {}

def receive_as_user(user_id, user_name):
    """Connect as a user, sync pending messages, and ACK them."""
    try:
        ws = ws_connect(WS_URL)

        # Connect
        ws.send(json.dumps({"type": "connect", "user_id": user_id}))
        connect_resp = json.loads(ws.recv())

        # Sync to get pending messages
        ws.send(json.dumps({"type": "sync", "user_id": user_id}))

        messages = []
        while True:
            resp = json.loads(ws.recv())
            if resp["type"] == "sync_complete":
                break
            if resp["type"] == "new_message":
                messages.append(resp)
                # ACK each message to mark it as delivered
                ws.send(json.dumps({"type": "ack", "message_id": resp["message_id"]}))
                json.loads(ws.recv())  # consume ack_ok response

        received_messages[user_name] = messages
        print(f"\u2705 {user_name} received {len(messages)} pending message(s)")
        for msg in messages:
            content = msg.get("content", "")[:60]
            print(f"   \U0001f4ac From user {msg.get('sender_id')}: {content}")

        ws.close()
    except Exception as e:
        print(f"\u274c {user_name} error: {e}")

# Run Bob and Charlie in parallel using threads
bob_thread = threading.Thread(target=receive_as_user, args=(2, "Bob"))
charlie_thread = threading.Thread(target=receive_as_user, args=(3, "Charlie"))

bob_thread.start()
charlie_thread.start()
bob_thread.join(timeout=10)
charlie_thread.join(timeout=10)

time.sleep(0.5)

print(f"\n\U0001f4ca Inbox status after Bob and Charlie ACKed:\n")
inbox_status = query("""
    SELECT u.display_name AS recipient, i.status, COUNT(*) as count
    FROM inbox i
    JOIN users u ON u.id = i.user_id
    JOIN messages m ON m.id = i.message_id
    WHERE m.chat_id = 3
    GROUP BY u.display_name, i.status
    ORDER BY u.display_name, i.status
""")
print(tabulate(inbox_status, headers="keys", tablefmt="simple_grid"))

print("\n\U0001f4a1 Bob and Charlie's messages are now 'delivered'.")
print("   Diana's messages are still 'pending' (she hasn't come online).")

📨 Connecting as Bob and Charlie to receive group messages...

✅ Bob received 1 pending message(s)
   💬 From user 1: Want to grab coffee later?



📊 Inbox status after Bob and Charlie ACKed:

┌───────────────┬───────────┬─────────┐
│ recipient     │ status    │   count │
├───────────────┼───────────┼─────────┤
│ Bob Smith     │ pending   │       1 │
├───────────────┼───────────┼─────────┤
│ Charlie Brown │ delivered │       1 │
├───────────────┼───────────┼─────────┤
│ Diana Prince  │ pending   │       4 │
└───────────────┴───────────┴─────────┘

💡 Bob and Charlie's messages are now 'delivered'.
   Diana's messages are still 'pending' (she hasn't come online).


## 🧩 Section 5: Partitioning Strategy — The Big Design Decision

When we use Redis pub/sub to deliver messages in real time, we need to decide how to organize our **channels**. This is one of the most important design decisions in a chat system.

### Two Approaches

**Option A: Partition by USER** (one channel per user)
```
Redis Channels:
  user:1  ← Alice subscribes here
  user:2  ← Bob subscribes here
  user:3  ← Charlie subscribes here

When Alice sends to Study Group (Bob, Charlie, Diana):
  PUBLISH user:2 message   → Bob
  PUBLISH user:3 message   → Charlie
  PUBLISH user:4 message   → Diana
  = 3 PUBLISH commands
```

**Option B: Partition by CHAT** (one channel per chat)
```
Redis Channels:
  chat:1  ← Alice & Bob subscribe (1:1 chat)
  chat:2  ← Alice & Charlie subscribe (1:1 chat)
  chat:3  ← Alice, Bob, Charlie, Diana subscribe (group)

When Alice sends to Study Group:
  PUBLISH chat:3 message
  = 1 PUBLISH command  (all subscribers receive it)
```

### 🤔 So Which is Better?

It depends on your workload! Let’s think with concrete numbers...

| Scenario | By User | By Chat |
|----------|---------|--------|
| User with 250 chats subscribes | **1 channel** | **250 channels** |
| Send 1:1 message | 1 PUBLISH | 1 PUBLISH |
| Send to 100-person group | **99 PUBLISHes** | **1 PUBLISH** |

> 💡 **Our server uses Partition by USER** — check `chat_server.py`, it publishes to `user:{user_id}` channels. This is the right choice because WhatsApp traffic is ~95% 1:1 messages.

Let’s run the numbers to see exactly why...

In [6]:
# ==== Partitioning Strategy: Back-of-Envelope Calculations ====
print("\U0001f9e9 Partitioning Strategy: By User vs By Chat\n")
print("=" * 60)

# --- Sample system parameters ---
total_users = 1_000
avg_chats_per_user = 250       # contacts/groups each user is in
pct_one_to_one = 0.95          # 95% of chats are 1:1
avg_group_size = 15            # average group has 15 members
messages_per_day = 100_000     # total messages across the system
pct_group_messages = 0.20      # 20% of messages go to groups

print("\U0001f4ca Sample System Parameters:")
print(f"   Users:              {total_users:,}")
print(f"   Avg chats/user:     {avg_chats_per_user}")
print(f"   1:1 vs group chats: {pct_one_to_one:.0%} / {1 - pct_one_to_one:.0%}")
print(f"   Avg group size:     {avg_group_size} members")
print(f"   Messages/day:       {messages_per_day:,}")
print(f"   Group messages:     {pct_group_messages:.0%}")

one_to_one_msgs = messages_per_day * (1 - pct_group_messages)
group_msgs = messages_per_day * pct_group_messages

# --- Strategy A: Partition by USER ---
print(f"\n{chr(9472) * 60}")
print("\U0001f4cc Strategy A: Partition by USER (one channel per user)")
print(f"{chr(9472) * 60}")

channels_a = total_users
subs_per_user_a = 1
total_subs_a = total_users * subs_per_user_a

# 1:1 -> 1 PUBLISH; group -> (avg_group_size - 1) PUBLISHes
publishes_1to1_a = one_to_one_msgs * 1
publishes_group_a = group_msgs * (avg_group_size - 1)
total_publishes_a = publishes_1to1_a + publishes_group_a

print(f"   Redis channels:       {channels_a:,}")
print(f"   Subscriptions/user:   {subs_per_user_a}")
print(f"   Total subscriptions:  {total_subs_a:,}")
print(f"   PUBLISHes/day (1:1):  {publishes_1to1_a:,.0f}")
print(f"   PUBLISHes/day (group):{publishes_group_a:,.0f}")
print(f"   TOTAL PUBLISHes/day:  {total_publishes_a:,.0f}")

# --- Strategy B: Partition by CHAT ---
print(f"\n{chr(9472) * 60}")
print("\U0001f4cc Strategy B: Partition by CHAT (one channel per chat)")
print(f"{chr(9472) * 60}")

total_chats = int(total_users * avg_chats_per_user / 2)  # shared between users
channels_b = total_chats
subs_per_user_b = avg_chats_per_user
total_subs_b = total_users * subs_per_user_b

# Every message -> 1 PUBLISH regardless of group size
total_publishes_b = messages_per_day

print(f"   Redis channels:       {channels_b:,}")
print(f"   Subscriptions/user:   {subs_per_user_b}")
print(f"   Total subscriptions:  {total_subs_b:,}")
print(f"   TOTAL PUBLISHes/day:  {total_publishes_b:,}")

# --- Side-by-side comparison ---
print(f"\n{chr(9472) * 60}")
print("\U0001f3c6 Comparison")
print(f"{chr(9472) * 60}")

comparison = [
    ["Redis channels",       f"{channels_a:,}",       f"{channels_b:,}"],
    ["Subscriptions/user",   str(subs_per_user_a),    str(subs_per_user_b)],
    ["Total subscriptions",  f"{total_subs_a:,}",     f"{total_subs_b:,}"],
    ["PUBLISHes/day",        f"{total_publishes_a:,.0f}", f"{total_publishes_b:,}"],
]
print(tabulate(comparison,
               headers=["Metric", "By User", "By Chat"],
               tablefmt="simple_grid"))

sub_ratio = total_subs_b / total_subs_a
pub_ratio = total_publishes_a / total_publishes_b

print(f"\n\U0001f4a1 Key Insight:")
print(f"   By-User needs {pub_ratio:.1f}x MORE publishes...")
print(f"   But By-Chat needs {sub_ratio:.0f}x MORE subscriptions!")
print(f"\n   Subscriptions cost memory on EVERY connected client.")
print(f"   For WhatsApp (1:1-heavy), fewer subscriptions = winner! \U0001f3c6")

🧩 Partitioning Strategy: By User vs By Chat

📊 Sample System Parameters:
   Users:              1,000
   Avg chats/user:     250
   1:1 vs group chats: 95% / 5%
   Avg group size:     15 members
   Messages/day:       100,000
   Group messages:     20%

────────────────────────────────────────────────────────────
📌 Strategy A: Partition by USER (one channel per user)
────────────────────────────────────────────────────────────
   Redis channels:       1,000
   Subscriptions/user:   1
   Total subscriptions:  1,000
   PUBLISHes/day (1:1):  80,000
   PUBLISHes/day (group):280,000
   TOTAL PUBLISHes/day:  360,000

────────────────────────────────────────────────────────────
📌 Strategy B: Partition by CHAT (one channel per chat)
────────────────────────────────────────────────────────────
   Redis channels:       125,000
   Subscriptions/user:   250
   Total subscriptions:  250,000
   TOTAL PUBLISHes/day:  100,000

────────────────────────────────────────────────────────────
🏆 Comparison
─

## 👥 Section 6: Adding and Removing Members

Group admins can add or remove members. Let’s see how this works at the database level and what **edge cases** we need to handle.

In [7]:
# ==== Managing Group Members ====
print("\U0001f465 Managing Group Members\n")
print("=" * 60)

# Show current members
print("\U0001f4cb Current Study Group members:\n")
members = query("""
    SELECT cp.user_id, u.display_name, cp.role
    FROM chat_participants cp
    JOIN users u ON u.id = cp.user_id
    WHERE cp.chat_id = 3
    ORDER BY cp.role DESC, cp.user_id
""")
print(tabulate(members, headers="keys", tablefmt="simple_grid"))

# --- Add Eve (user_id=5) ---
print("\n\u2795 Adding Eve (user_id=5) to Study Group...\n")
conn = get_db()
try:
    cur = conn.cursor()
    cur.execute("""
        INSERT INTO chat_participants (chat_id, user_id, role)
        VALUES (3, 5, 'member')
        ON CONFLICT (chat_id, user_id) DO NOTHING
    """)
    conn.commit()
    cur.close()
finally:
    conn.close()

members_after = query("""
    SELECT cp.user_id, u.display_name, cp.role
    FROM chat_participants cp
    JOIN users u ON u.id = cp.user_id
    WHERE cp.chat_id = 3
    ORDER BY cp.role DESC, cp.user_id
""")
print(tabulate(members_after, headers="keys", tablefmt="simple_grid"))
print(f"\n\u2705 Eve added! Group now has {len(members_after)} members")

# --- Remove Eve ---
print("\n\n\u2796 Removing Eve from Study Group...\n")
conn = get_db()
try:
    cur = conn.cursor()
    cur.execute(
        "DELETE FROM chat_participants WHERE chat_id = 3 AND user_id = 5"
    )
    conn.commit()
    cur.close()
finally:
    conn.close()

members_final = query("""
    SELECT cp.user_id, u.display_name, cp.role
    FROM chat_participants cp
    JOIN users u ON u.id = cp.user_id
    WHERE cp.chat_id = 3
    ORDER BY cp.role DESC, cp.user_id
""")
print(tabulate(members_final, headers="keys", tablefmt="simple_grid"))
print(f"\n\u2705 Eve removed! Group back to {len(members_final)} members")

# --- Edge cases ---
print("\n\n\U0001f914 Edge Cases to Think About:")
print("\u2500" * 60)
print("""
1. \U0001f504 In-flight messages: If Alice sends while Eve is being removed,
   Eve might still receive that message. This is OK \u2014 eventual
   consistency. The NEXT message won't be delivered to her.

2. \U0001f4dc Message history: Should Eve still see old messages?
   WhatsApp keeps messages the user received before removal.
   Our inbox entries already created won't be deleted.

3. \U0001f451 Last admin leaves: What if the only admin is removed?
   Option A: Auto-promote the oldest member to admin
   Option B: Prevent the last admin from leaving
   Option C: Dissolve the group

4. \U0001f514 Member notifications: Existing members should see a
   system message like "Eve was added" or "Eve left".
   This would be a message with message_type = 'system'.
""")

👥 Managing Group Members

📋 Current Study Group members:

┌───────────┬────────────────┬────────┐
│   user_id │ display_name   │ role   │
├───────────┼────────────────┼────────┤
│         2 │ Bob Smith      │ member │
├───────────┼────────────────┼────────┤
│         3 │ Charlie Brown  │ member │
├───────────┼────────────────┼────────┤
│         4 │ Diana Prince   │ member │
├───────────┼────────────────┼────────┤
│         1 │ Alice Johnson  │ admin  │
└───────────┴────────────────┴────────┘

➕ Adding Eve (user_id=5) to Study Group...

┌───────────┬────────────────┬────────┐
│   user_id │ display_name   │ role   │
├───────────┼────────────────┼────────┤
│         2 │ Bob Smith      │ member │
├───────────┼────────────────┼────────┤
│         3 │ Charlie Brown  │ member │
├───────────┼────────────────┼────────┤
│         4 │ Diana Prince   │ member │
├───────────┼────────────────┼────────┤
│         5 │ Eve Wilson     │ member │
├───────────┼────────────────┼────────┤
│         1 │ Ali

## 📈 Section 7: Scaling Groups — The Hard Problems

### The 100-Member Limit

Our `chats` table has `max_participants = 100`. Why not unlimited?

```
📊 Cost of One Message to a Big Group (back-of-envelope)

Group Size │ Inbox Writes │ Redis PUBLISHes │ Est. Latency
───────────┼──────────────┼─────────────────┼─────────────
     10    │       9      │        9        │    ~9ms
     50    │      49      │       49        │   ~49ms
    100    │      99      │       99        │   ~99ms
    500    │     499      │      499        │  ~499ms
  1,000    │     999      │      999        │  ~999ms
 10,000    │   9,999      │    9,999        │ ~10 sec ⚠️
```

> ⚠️ At 10,000 members, a single message takes **~10 seconds** just for inbox writes. Users would see massive delays!

### 🌟 The Celebrity Problem

Imagine a famous person creates a group with 10,000 followers. Every message they send:
- Creates **9,999 inbox rows** in the database
- Triggers **9,999 Redis PUBLISH commands**
- All hitting the **same partition** (if sharded by chat_id)

This is called the **“hot partition”** or **“celebrity problem”** — one entity generating disproportionate load.

### 🔄 Adaptive Partitioning

Smart systems use a **hybrid approach** based on group size:

```
                     Group Size
                         │
           ┌─────────────┼─────────────┐
           │             │             │
      Small (≤50)   Medium (50-500)   Large (500+)
           │             │             │
     Partition by   Partition by     Broadcast
     USER channel   CHAT channel     channel +
     (fan-out on    (fan-out on      batched
      write)         read)           delivery
```

- **Small groups (≤50):** Fan-out on write to `user:N` channels. Fast and simple.
- **Medium groups (50-500):** Switch to a `chat:N` channel. All members subscribe. Less write amplification.
- **Large groups (500+):** Dedicated broadcast infrastructure with **batched delivery** — users pull messages when they come online instead of push.

### 💡 WhatsApp’s Approach

WhatsApp caps groups at 1,024 members because:
1. **Fan-out cost stays manageable** at this size
2. **End-to-end encryption** requires encrypting for each member’s unique key
3. **No celebrity accounts** — WhatsApp is for personal communication
4. For broadcast needs, they have **WhatsApp Channels** (a separate system)

## 🧹 Cleanup

Remove any data we created during this notebook so the database is back to its original state.

In [8]:
# ==== Clean up data created during this notebook ====
print("\U0001f9f9 Cleaning up...\n")

conn = get_db()
try:
    cur = conn.cursor()

    # Remove the "Project Team" group we created in Section 2
    cur.execute("""
        DELETE FROM chat_sequences WHERE chat_id IN (
            SELECT id FROM chats WHERE name = 'Project Team'
        )
    """)
    cur.execute("""
        DELETE FROM chat_participants WHERE chat_id IN (
            SELECT id FROM chats WHERE name = 'Project Team'
        )
    """)
    cur.execute("DELETE FROM chats WHERE name = 'Project Team'")

    # Remove the test message we sent to Study Group
    cur.execute("""
        DELETE FROM inbox WHERE message_id IN (
            SELECT id FROM messages
            WHERE chat_id = 3
            AND content LIKE '%%study together tonight%%'
        )
    """)
    cur.execute("""
        DELETE FROM messages
        WHERE chat_id = 3
        AND content LIKE '%%study together tonight%%'
    """)

    # Ensure Eve is not in Study Group (safety check)
    cur.execute(
        "DELETE FROM chat_participants WHERE chat_id = 3 AND user_id = 5"
    )

    conn.commit()
    cur.close()
    print("\u2705 Removed 'Project Team' chat and its participants")
    print("\u2705 Removed test message and inbox entries from Study Group")
    print("\u2705 Ensured Eve is not in Study Group")
finally:
    conn.close()

print("\n\U0001f389 All clean! Database is back to its original state.")

🧹 Cleaning up...

✅ Removed 'Project Team' chat and its participants
✅ Removed test message and inbox entries from Study Group
✅ Ensured Eve is not in Study Group

🎉 All clean! Database is back to its original state.


## 📝 Summary

### What We Learned

| Concept | Key Takeaway |
|---------|-------------|
| **Unified Design** | 1:1 chats and groups use the same code path — a 1:1 chat is just a group of 2 |
| **Fan-Out on Write** | One message → N-1 inbox rows + N-1 Redis publishes |
| **Write Amplification** | Cost grows linearly with group size |
| **Partition by User** | Best for 1:1-heavy workloads (like WhatsApp): 1 channel per user |
| **Partition by Chat** | Better for large groups: 1 channel per chat, but many more subscriptions |
| **Admin Controls** | Simple INSERT/DELETE on `chat_participants` with edge case handling |
| **Group Size Limits** | Keep groups small to limit fan-out cost; use adaptive partitioning for larger ones |

### 🔑 Key Formula

```
Fan-out cost per message = (group_size - 1) × (inbox_write + redis_publish)
```

For a **100-member group** sending 1 message/minute:
- 99 inbox writes/minute = **5,940/hour** = **142,560/day**
- That’s just **one** active group!

### ➡️ Next Up: Notebook 4 — Encryption Basics

In the final notebook, we’ll explore how WhatsApp implements **end-to-end encryption** so that even the server can’t read your messages:
- Public/private key pairs
- Encrypting messages per recipient
- Why encryption makes group messaging even harder (N encryptions per message!)